> **Save your own copy first.** Unless this notebook is already in your own Google Drive, Colab does not keep your changes. Go to **File → Save a copy in Drive** and work in the copy that opens in a new tab. You find it later in Google Drive, in *Colab Notebooks*.

![Cultural Data Analysis · Week 3: Structured Data and Machine Learning](https://goto4711.github.io/cda-teaching-tools/assets/notebooks/week3-banner.png)

# Data critique: from a table to a label

This notebook goes with the ten questions of the [data critique](https://goto4711.github.io/cda-teaching-tools/workshops/week3/#critique). The example are the donation records of *Global Aid Alliance*. Read the data sheet on the week 3 code-along webiste.

Write your answers in the [data critique template](https://github.com/goto4711/cda-teaching-tools/blob/main/notebooks/templates/data_critique_template.md), a Markdown file you keep for your projects. Use this notebook to test what you write and provide evidence.

This week is all about how a machine learns from a table. What would it predict (the **label**), whom does that label favour, where do the **features** come from, and can a harmless-looking column stand in for a sensitive one?

Run the cell below to load the data.

In [ ]:
#Keep cell
import pandas as pd

# A web address, or a file you uploaded to Colab (folder symbol on the left), e.g. 'my_data.csv'
DATA_FILE = 'https://raw.githubusercontent.com/goto4711/cdai/main/donor_records_export.csv'

if DATA_FILE.endswith(('.xlsx', '.xls')):
  data = pd.read_excel(DATA_FILE)
else:
  data = pd.read_csv(DATA_FILE)       # for a file separated by semicolons: pd.read_csv(DATA_FILE, sep=';')
print(len(data), 'rows,', len(data.columns), 'columns')
data.head()

---

## 1. Make the label (question 5)

The NGO wants to keep donors giving. A model that predicts which donors stay needs a column that says, for every donor, whether they stayed: the **label**. The data has no such column. Try to define it and discuss how it is a **proxy** for "loyalty".

The next cell makes three labels, one row per donor:

* **A:** gave in at least two different years;
* **B:** gave again within 12 months of an earlier gift;
* **C:** at least one gift is marked as a recurring gift (`recurring_gift`).

Change the definitions, or add your own. Note that a donor is whatever the column `donor_id` stands for: compare the column `names` (how many different names each ID has).

Run the next cell.

In [ ]:
ID, DATE = 'donor_id', 'donation_date'

# Turn the text in the date column into dates Python can calculate with (a date it cannot read becomes NaT, "not a time")
data['date'] = pd.to_datetime(data[DATE], errors='coerce')

# Three definitions of a loyal donor. Each function gets all the gifts of one donor and answers True or False.
def loyal_A(gifts):
  # A: gave in at least two different years
  years = [d.year for d in gifts['date'] if not pd.isna(d)]
  return len(set(years)) >= 2

def loyal_B(gifts):
  # B: gave again within 12 months (365 days) of an earlier gift
  dates = sorted([d for d in gifts['date'] if not pd.isna(d)])
  for i in range(1, len(dates)):
    if (dates[i] - dates[i - 1]).days <= 365:
      return True
  return False

def loyal_C(gifts):
  # C: at least one gift marked as recurring (an answer that starts with 'yes')
  for answer in gifts['recurring_gift']:
    if str(answer).strip()[:3] in ['Yes', 'yes', 'YES']:
      return True
  return False

# One row per donor
rows = []
for donor, gifts in data.groupby(ID):
  rows.append({
    ID: donor,
    'gifts': len(gifts),
    'years': len(set([d.year for d in gifts['date'] if not pd.isna(d)])),
    'names': len(gifts['donor_name'].dropna().unique()),
    'loyal_A': loyal_A(gifts),
    'loyal_B': loyal_B(gifts),
    'loyal_C': loyal_C(gifts),
  })
donors = pd.DataFrame(rows)
LABELS = ['loyal_A', 'loyal_B', 'loyal_C']

print(len(donors), 'donors. Share counted as loyal:')
print(donors[LABELS].mean().round(2))
print('\nDo the labels agree? (A against B: how many donors for each combination)')
print(pd.crosstab(donors['loyal_A'], donors['loyal_B']))
donors.head()

How many donors are "loyal" depends on the definition. Which donors change sides? What does each definition count, and what does it miss? 

---

## 2. Whom does the label favour? (questions 5, 9)

A model learns to find donors like the ones its label calls loyal. Who are they? The next cell gives the share of loyal donors per group, for each label. Each donor gets the value they have most often in `GROUP`. Try `'channel'`, `'gender'`, `'age_range'` and `'region'`.

Run the next cell.

In [ ]:
GROUP = 'channel'

# A donor's usual value in GROUP: the one in most of their gifts (the first in the alphabet if there is a tie)
def usual(values):
  values = [v for v in values if not pd.isna(v)]
  best, best_count = '(missing)', 0
  for v in sorted(set(values)):
    count = len([w for w in values if w == v])
    if count > best_count:
      best, best_count = v, count
  return best

group_of_donor = []
for donor, gifts in data.groupby(ID):
  group_of_donor.append(usual(gifts[GROUP]))
donors[GROUP] = group_of_donor

table = donors.groupby(GROUP)[LABELS].mean().round(2)
table['donors'] = donors[GROUP].value_counts()
table

Which groups would the fundraising target? Is a difference real, or a trace of how the data was made, e.g. of how gifts by post are recorded? Small groups give unstable shares: look at the column `donors`.

---

## 3. Where do the features come from? (question 6)

The other columns become the **features** the model learns from. Who filled them in? If that differs by channel, the model may learn the channel rather than the donor. The next cell shows the share of each value of `COLUMN` per value of `SOURCE`, with missing values as `(missing)`.

Run the next cell.

In [ ]:
SOURCE, COLUMN = 'channel', 'gender'

# Count the rows for each pair of values; '(missing)' stands for empty cells, and normalize='index' turns each row into shares
pd.crosstab(data[SOURCE].fillna('(missing)'), data[COLUMN].fillna('(missing)'), normalize='index').round(2)

---

## 4. Proxies (questions 8, 9)

Leaving a sensitive column out does not keep it out of the model if another column gives it away. In real data, a postcode can stand for income or ethnicity, a first name for gender, a campaign for religion. The next cell shows how well `FEATURE` predicts `SENSITIVE`: for each value of `FEATURE`, the share of each value of `SENSITIVE`. The more these rows differ from one another, the more the feature gives away.

Run the next cell.

In [ ]:
FEATURE, SENSITIVE = 'campaign', 'age_range'

# Count the rows for each pair of values; '(missing)' stands for empty cells, and normalize='index' turns each row into shares
pd.crosstab(data[FEATURE].fillna('(missing)'), data[SENSITIVE].fillna('(missing)'), normalize='index').round(2)

In your own data, think first about which columns *could* stand in for a sensitive one, and check those.

**Your own checks**